## Resume Surfer

In [ ]:
%pip install -q langchain-ollama


In [ ]:
import os


import gradio as gr
from openai import OpenAI

import tiktoken
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
# Constants
load_dotenv(override=True)

MODEL_OPENAI = "gpt-4o-mini"
LOCAL_MODEL_OPENAI = "gpt-oss:20b"


db_name = "resume_db"

In [ ]:
def read_pdf_file(file_path):
    if file_path.endswith(".pdf"):
        loader = PyPDFLoader(file_path)

        resume = loader.load()

        return resume
    else:
        return None

In [ ]:
resume_path = input("Enter the path to the resume file: ")
resume = read_pdf_file(resume_path)

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are an HR assistant, friendly assistant representing the recruiting firm.
You're chatting with a client and asked about a candidate
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
encoding = tiktoken.encoding_for_model(MODEL_OPENAI)

resume_text = "\n".join(doc.page_content for doc in resume)

tokens = encoding.encode(resume_text)
token_count = len(tokens)
print(f"Total tokens for {MODEL_OPENAI}: {token_count:,}")

In [ ]:
def load_to_store():
    if resume is None:
        raise ValueError("Resume not loaded")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    
    chunks = text_splitter.split_documents(resume)

    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    if os.path.exists(db_name):
        Chroma(
            persist_directory=db_name,
            embedding_function=embeddings
        ).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_name
    )

    print(f"Vectorstore created with {vectorstore._collection.count()} documents")

    return vectorstore


In [ ]:
# Let's investigate the vectors

vectorstore = load_to_store()

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(model=LOCAL_MODEL_OPENAI, temperature=0)

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
gr.ChatInterface(answer_question).launch()